# <font color="#F48E16">XAI aplicada al Deep Learning: XAI aplicada a Imagenet</font>

Material generado por <a href="https://www.linkedin.com/in/christian-oliva-moya-ingeniero/">Christian Oliva</a>. Cualquier duda, sugerencia o errata, no duden en contactar.

**Versión 1.0** - 12 de mayo de 2026

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tqdm import tqdm
from time import time
from sklearn.linear_model import LinearRegression

import os
import urllib.request
import tarfile

url = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2.tgz"
download_path = "./datasets/imagenette/imagenette2.tgz"
extract_path = "./datasets/imagenette/"

os.makedirs(extract_path, exist_ok=True)

#urllib.request.urlretrieve(url, download_path)

#with tarfile.open(download_path) as tar:
#    tar.extractall(path=extract_path)

print("Dataset listo en:", extract_path)

In [ ]:
from tensorflow.keras.applications.mobilenet import preprocess_input

train_ds = tf.keras.utils.image_dataset_from_directory(
    "./datasets/imagenette/imagenette2/train",
    image_size=(224,224),
    batch_size=128
)
train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y))

val_ds = tf.keras.utils.image_dataset_from_directory(
    "./datasets/imagenette/imagenette2/val",
    image_size=(224,224),
    batch_size=128
)
val_ds = val_ds.map(lambda x, y: (preprocess_input(x), y))

In [ ]:
x_train, y_train = next(iter(train_ds.unbatch().take(20).batch(20)))
x_train = x_train.numpy()
y_train = y_train.numpy()
x_train.shape, y_train.shape

In [ ]:
import json

# Descargamos el índice de clases de ImageNet (PyTorch)
url = "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json"
with urllib.request.urlopen(url) as f:
    imagenet_class_index = json.load(f)  # dict: str(index) -> [wnid, label]

# Mapeo Imagenette -> WNID
imagenette_to_wnid = {
    0: "n01440764",
    1: "n02102040",
    2: "n02979186",
    3: "n03000684",
    4: "n03028079",
    5: "n03394916",
    6: "n03417042",
    7: "n03425413",
    8: "n03445777",
    9: "n03888257"
}

# Crear diccionario completo: índice Imagenette → (imagenet_index, wnid, nombre)
imagenette_to_imagenet = {}
for idx_imagenette, wnid in imagenette_to_wnid.items():
    # Buscar índice en ImageNet por wnid
    idx_imagenet = next(
        int(k) for k, v in imagenet_class_index.items()
        if v[0] == wnid
    )
    class_name = imagenet_class_index[str(idx_imagenet)][1]
    imagenette_to_imagenet[idx_imagenette] = {
        "imagenet_index": idx_imagenet,
        "wnid": wnid,
        "name": class_name
    }

# Ejemplo de uso
for i in range(10):
    print(f"Imagenette {i} -> {imagenette_to_imagenet[i]}")

In [ ]:
new_y_train = []
for y in y_train:
    new_y_train.append(imagenette_to_imagenet[y]["imagenet_index"])
y_train = np.array(new_y_train)
y_train

In [ ]:
def build_function():
    return tf.keras.applications.MobileNet(weights="imagenet")

In [ ]:
## FUNCIONES DEL MAIN DE KERAS
## SOLO SE HA CAMBIADO DE ESTA CELDA layers.RELU6 por layers.Activation(d_f(tf.nn.relu))
from keras.src.applications import imagenet_utils
from keras.src import backend
from keras.src import layers
from keras.src.models import Functional
BASE_WEIGHT_PATH = (
    "https://storage.googleapis.com/tensorflow/keras-applications/mobilenet/"
)
from keras.src.utils import file_utils

def build_manual_mobilenet(d_f):

    def MobileNet(
        input_shape=None,
        alpha=1.0,
        depth_multiplier=1,
        dropout=1e-3,
        include_top=True,
        weights="imagenet",
        input_tensor=None,
        pooling=None,
        classes=1000,
        classifier_activation="softmax",
        name=None,
    ):
        if not (weights in {"imagenet", None} or file_utils.exists(weights)):
            raise ValueError(
                "The `weights` argument should be either "
                "`None` (random initialization), 'imagenet' "
                "(pre-training on ImageNet), "
                "or the path to the weights file to be loaded. "
                f"Received weights={weights}"
            )

        if weights == "imagenet" and include_top and classes != 1000:
            raise ValueError(
                "If using `weights='imagenet'` with `include_top=True`, "
                "`classes` should be 1000.  "
                f"Received classes={classes}"
            )

        # Determine proper input shape and default size.
        if input_shape is None:
            default_size = 224
        else:
            if backend.image_data_format() == "channels_first":
                rows = input_shape[1]
                cols = input_shape[2]
            else:
                rows = input_shape[0]
                cols = input_shape[1]

            if rows == cols and rows in [128, 160, 192, 224]:
                default_size = rows
            else:
                default_size = 224

        input_shape = imagenet_utils.obtain_input_shape(
            input_shape,
            default_size=default_size,
            min_size=32,
            data_format=backend.image_data_format(),
            require_flatten=include_top,
            weights=weights,
        )

        if backend.image_data_format() == "channels_last":
            row_axis, col_axis = (0, 1)
        else:
            row_axis, col_axis = (1, 2)
        rows = input_shape[row_axis]
        cols = input_shape[col_axis]

        if weights == "imagenet":
            if depth_multiplier != 1:
                raise ValueError(
                    "If imagenet weights are being loaded, "
                    "depth multiplier must be 1.  "
                    f"Received depth_multiplier={depth_multiplier}"
                )

            if alpha not in [0.25, 0.50, 0.75, 1.0]:
                raise ValueError(
                    "If imagenet weights are being loaded, "
                    "alpha can be one of"
                    "`0.25`, `0.50`, `0.75` or `1.0` only.  "
                    f"Received alpha={alpha}"
                )

            if rows != cols or rows not in [128, 160, 192, 224]:
                rows = 224
                warnings.warn(
                    "`input_shape` is undefined or non-square, "
                    "or `rows` is not in [128, 160, 192, 224]. "
                    "Weights for input shape (224, 224) will be "
                    "loaded as the default.",
                    stacklevel=2,
                )

        if input_tensor is None:
            img_input = layers.Input(shape=input_shape)
        else:
            if not backend.is_keras_tensor(input_tensor):
                img_input = layers.Input(tensor=input_tensor, shape=input_shape)
            else:
                img_input = input_tensor

        x = _conv_block(img_input, 32, alpha, strides=(2, 2))
        x = _depthwise_conv_block(x, 64, alpha, depth_multiplier, block_id=1)

        x = _depthwise_conv_block(
            x, 128, alpha, depth_multiplier, strides=(2, 2), block_id=2
        )
        x = _depthwise_conv_block(x, 128, alpha, depth_multiplier, block_id=3)

        x = _depthwise_conv_block(
            x, 256, alpha, depth_multiplier, strides=(2, 2), block_id=4
        )
        x = _depthwise_conv_block(x, 256, alpha, depth_multiplier, block_id=5)

        x = _depthwise_conv_block(
            x, 512, alpha, depth_multiplier, strides=(2, 2), block_id=6
        )
        x = _depthwise_conv_block(x, 512, alpha, depth_multiplier, block_id=7)
        x = _depthwise_conv_block(x, 512, alpha, depth_multiplier, block_id=8)
        x = _depthwise_conv_block(x, 512, alpha, depth_multiplier, block_id=9)
        x = _depthwise_conv_block(x, 512, alpha, depth_multiplier, block_id=10)
        x = _depthwise_conv_block(x, 512, alpha, depth_multiplier, block_id=11)

        x = _depthwise_conv_block(
            x, 1024, alpha, depth_multiplier, strides=(2, 2), block_id=12
        )
        x = _depthwise_conv_block(x, 1024, alpha, depth_multiplier, block_id=13)

        if include_top:
            x = layers.GlobalAveragePooling2D(keepdims=True)(x)
            x = layers.Dropout(dropout, name="dropout")(x)
            x = layers.Conv2D(classes, (1, 1), padding="same", name="conv_preds")(x)
            x = layers.Reshape((classes,), name="reshape_2")(x)
            imagenet_utils.validate_activation(classifier_activation, weights)
            x = layers.Activation(
                activation=classifier_activation, name="predictions"
            )(x)
        else:
            if pooling == "avg":
                x = layers.GlobalAveragePooling2D()(x)
            elif pooling == "max":
                x = layers.GlobalMaxPooling2D()(x)

        # Ensure that the model takes into account
        # any potential predecessors of `input_tensor`.
        if input_tensor is not None:
            inputs = operation_utils.get_source_inputs(input_tensor)
        else:
            inputs = img_input

        # Create model.
        if name is None:
            name = f"mobilenet_{alpha:0.2f}_{rows}"
        model = Functional(inputs, x, name=name)

        # Load weights.
        if weights == "imagenet":
            if alpha == 1.0:
                alpha_text = "1_0"
            elif alpha == 0.75:
                alpha_text = "7_5"
            elif alpha == 0.50:
                alpha_text = "5_0"
            else:
                alpha_text = "2_5"

            if include_top:
                model_name = "mobilenet_%s_%d_tf.h5" % (alpha_text, rows)
                weight_path = BASE_WEIGHT_PATH + model_name
                weights_path = file_utils.get_file(
                    model_name, weight_path, cache_subdir="models"
                )
            else:
                model_name = "mobilenet_%s_%d_tf_no_top.h5" % (alpha_text, rows)
                weight_path = BASE_WEIGHT_PATH + model_name
                weights_path = file_utils.get_file(
                    model_name, weight_path, cache_subdir="models"
                )
            model.load_weights(weights_path)
        elif weights is not None:
            model.load_weights(weights)

        return model

    def _depthwise_conv_block(
        inputs,
        pointwise_conv_filters,
        alpha,
        depth_multiplier=1,
        strides=(1, 1),
        block_id=1,
    ):
        channel_axis = 1 if backend.image_data_format() == "channels_first" else -1
        pointwise_conv_filters = int(pointwise_conv_filters * alpha)

        if strides == (1, 1):
            x = inputs
        else:
            x = layers.ZeroPadding2D(
                ((0, 1), (0, 1)), name="conv_pad_%d" % block_id
            )(inputs)
        x = layers.DepthwiseConv2D(
            (3, 3),
            padding="same" if strides == (1, 1) else "valid",
            depth_multiplier=depth_multiplier,
            strides=strides,
            use_bias=False,
            name="conv_dw_%d" % block_id,
        )(x)
        x = layers.BatchNormalization(
            axis=channel_axis, name="conv_dw_%d_bn" % block_id
        )(x)
        # x = layers.ReLU(6.0, name="conv_dw_%d_relu" % block_id)(x) # OLD
        x = layers.Activation(d_f(tf.nn.relu6), name="conv_dw_%d_relu" % block_id)(x) # NEW

        x = layers.Conv2D(
            pointwise_conv_filters,
            (1, 1),
            padding="same",
            use_bias=False,
            strides=(1, 1),
            name="conv_pw_%d" % block_id,
        )(x)
        x = layers.BatchNormalization(
            axis=channel_axis, name="conv_pw_%d_bn" % block_id
        )(x)
        # return layers.ReLU(6.0, name="conv_pw_%d_relu" % block_id)(x) # OLD
        return layers.Activation(d_f(tf.nn.relu6), name="conv_pw_%d_relu" % block_id)(x) # NEW

    def _conv_block(inputs, filters, alpha, kernel=(3, 3), strides=(1, 1)):
        channel_axis = 1 if backend.image_data_format() == "channels_first" else -1
        filters = int(filters * alpha)
        x = layers.Conv2D(
            filters,
            kernel,
            padding="same",
            use_bias=False,
            strides=strides,
            name="conv1",
        )(inputs)
        x = layers.BatchNormalization(axis=channel_axis, name="conv1_bn")(x)
        # return layers.ReLU(6.0, name="conv1_relu")(x) # OLD
        return layers.Activation(d_f(tf.nn.relu6), name="conv1_relu")(x) # NEW

    return MobileNet(weights="imagenet")

In [ ]:
model = build_function()
model.compile(loss="sparse_categorical_crossentropy", metrics=["Accuracy"])
model.summary()
model.evaluate(x_train, y_train)

### <font color="#F48E16">Gradient x Input</font>

In [ ]:
def gradient(model, X, y):
  X_tf = tf.convert_to_tensor(X, dtype=tf.float32)

  with tf.GradientTape() as tape:
    tape.watch(X_tf)
    y_pred = model(X_tf)

    # Seleccionamos la salida correspondiente a la clase real
    indices = tf.stack([tf.range(tf.shape(y_pred)[0]), y], axis=1)
    y_class = tf.gather_nd(y_pred, indices)

  grads = tape.gradient(y_class, X_tf)
  return grads.numpy()

In [ ]:
def gradientxinput(model, X, y):
  return gradient(model, X, y) * X

### <font color="#F48E16">LRP</font>

In [ ]:
def lrp(f):

    @tf.custom_gradient
    def f_custom_grad(x):
        output = f(x)

        def backward(dy):
            custom_grad = f(x)/x
            return dy * custom_grad

        return output, backward
    return f_custom_grad

In [ ]:
model_lrp = build_manual_mobilenet(lrp)
model_lrp.compile(loss="sparse_categorical_crossentropy", metrics=["Accuracy"])
#model_lrp.summary()
model_lrp.evaluate(x_train, y_train)

## <font color="#F48E16">EPC</font>

In [ ]:
from tqdm import tqdm
def epc(X, y, model, Rx, X_blur=None, use_blur=False, local=False, verbose=False):
  at = Rx.shape[-1]
  m = model.evaluate(X, y, batch_size=len(X), verbose=False)[-1]
  mmas_all = []
  mmenos_all = []
  pred_mmas_all = []
  pred_mmenos_all = []
  iterador_at = tqdm(range(0, at, 1506)) if verbose else range(0, at, 1506) # Voy de 1506 en 1506 para acelerar y tener 100 pasos
  if not local: # Si es global
    for i in iterador_at:
        ar = at - i
        factor1 = (at - ar) / at
        umbral = np.sort(Rx)[i] # Usamos el mismo umbral para todo el dataset

        # Prueba 1
        X_prueba1 = X.copy()
        if use_blur:
            X_prueba1[:, Rx.reshape(224, 224, 3) <= umbral] = X_blur[:, Rx.reshape(224, 224, 3) <= umbral]
        else:
            X_prueba1[:, Rx.reshape(224, 224, 3) <= umbral] = 0
        mmas = model.evaluate(X_prueba1, y, batch_size=len(X_prueba1), verbose=False)[-1]

        # Prueba 2
        X_prueba2 = X.copy()
        if use_blur:
            X_prueba2[:, Rx.reshape(224, 224, 3) >= umbral] = X_blur[:, Rx.reshape(224, 224, 3) >= umbral]
        else:
            X_prueba2[:, Rx.reshape(224, 224, 3) >= umbral] = 0
        mmenos = model.evaluate(X_prueba2, y, batch_size=len(X_prueba2), verbose=False)[-1]

        mmas_all.append(mmas)
        mmenos_all.append(mmenos)

  else: # Si es local
    for i in iterador_at:
        ar = at - i
        factor1 = (at - ar) / at

        # Prueba 1
        X_prueba1 = X.copy()
        for j, (x, r) in enumerate(zip(X, Rx)): # Por cada instancia x hay una relevancia r
            umbral = np.sort(r)[i] # y tenemos que usar esa r para definir el umbral
            if use_blur:
                X_prueba1[j, r.reshape(224, 224, 3) <= umbral] = X_blur[j, r.reshape(224, 224, 3) <= umbral]
            else:
                X_prueba1[j, r.reshape(224, 224, 3) <= umbral] = 0
        mmas = model.evaluate(X_prueba1, y, batch_size=len(X_prueba1), verbose=False)[-1]
        pred_mmas = model.predict(X_prueba1, batch_size=len(X_prueba1), verbose=False) # (100, 1000)
        pred_mmas_all.append(pred_mmas)

        # Prueba 2
        X_prueba2 = X.copy()
        for j, (x, r) in enumerate(zip(X, Rx)): # Lo mismo aquí con la prueba 2
            umbral = np.sort(r)[i]
            if use_blur:
                X_prueba2[j, r.reshape(224, 224, 3) >= umbral] = X_blur[j, r.reshape(224, 224, 3) >= umbral]
            else:
                X_prueba2[j, r.reshape(224, 224, 3) >= umbral] = 0
        mmenos = model.evaluate(X_prueba2, y, batch_size=len(X_prueba2), verbose=False)[-1]
        pred_mmenos = model.predict(X_prueba2, batch_size=len(X_prueba2), verbose=False) # (100, 1000)
        pred_mmenos_all.append(pred_mmenos)

        factor2 = (mmas - mmenos) / m

        mmas_all.append(mmas)
        mmenos_all.append(mmenos)

  return np.array(mmas_all), np.array(mmenos_all), np.array(pred_mmas_all), np.array(pred_mmenos_all)

In [ ]:
# https://arxiv.org/abs/2004.03383 utiliza blur para imagenes
# https://arxiv.org/abs/1604.04004 tambien analiza el blur para detectar ataques contra imagenes

import cv2

x_train_blur = np.array([np.clip(cv2.GaussianBlur(x, (15, 15), 0), -1, 1) for x in x_train])

# Articulo que motiva insertion y deletion
# https://arxiv.org/abs/1806.07421
# Algunas metricas que evaluan insertion y deletion. https://arxiv.org/pdf/2201.13291
# LIMITACIONES DE insertion deletion
# blur vs black baseline cambia resultados
# el ranking domina más que el valor real
# hay dependencia fuerte del tipo de perturbación

In [ ]:
plt.subplot(1,2,1)
plt.imshow((x_train[0]+1)/2)
plt.subplot(1,2,2)
plt.imshow((x_train_blur[0]+1)/2)

In [ ]:
ACC = model.evaluate(x_train, y_train)[1]
ACC

In [ ]:
ACC_BLUR = model.evaluate(x_train_blur, y_train)[1]
ACC_BLUR

## <font color="#F48E16">Usando media = 0 como baseline</font>

In [ ]:
model_name = "MOBILENET"
activation = "relu"
l1reg = 0.0
use_blur = False

R_gradientxinput = gradientxinput(model, x_train, y_train)
R_lrp = gradientxinput(model_lrp, x_train, y_train)

(mmas_gradientxinput,
 mmenos_gradientxinput,
 pred_mmas_gradientxinput,
 pred_mmenos_gradientxinput) = epc(x_train,
                                   y_train,
                                   model,
                                   R_gradientxinput.reshape(-1, 224*224*3),
                                   X_blur=x_train_blur,
                                   use_blur=use_blur,
                                   local=True,
                                   verbose=True)

# LRP
(mmas_lrp,
 mmenos_lrp,
 pred_mmas_lrp,
 pred_mmenos_lrp) = epc(x_train,
                        y_train,
                        model,
                        R_lrp.reshape(-1, 224*224*3),
                        X_blur=x_train_blur,
                        use_blur=use_blur,
                        local=True,
                        verbose=True)


print(" > DONE")

In [ ]:
at = 224*224*3
iterador_at = np.arange(0, at, 1506) # Voy de 1506 en 1506 para acelerar y tener 100 pasos
ar = at - iterador_at
factor1 = (at - ar) / at

In [ ]:
epc_gradientxinput = factor1 * (mmas_gradientxinput - mmenos_gradientxinput) / ACC
imax = np.argmax(epc_gradientxinput)

plt.figure(figsize=(15,3))
plt.subplot(1,2,1)
plt.plot(mmas_gradientxinput, color="green", label="mmas")
plt.plot(imax, mmas_gradientxinput[imax], 'o', color="green")
plt.plot(mmenos_gradientxinput, color="blue", label="mmenos")
plt.plot(imax, mmenos_gradientxinput[imax], 'o', color="blue")
plt.axvline(imax, color="gray", alpha=0.2)
plt.legend()
plt.subplot(1,2,2)
plt.plot(epc_gradientxinput, color="red", label="EPC")
plt.plot(imax, epc_gradientxinput[imax], 'o', color="red")
plt.legend()
plt.show()

In [ ]:
epc_lrp = factor1 * (mmas_lrp - mmenos_lrp) / ACC
imax = np.argmax(epc_lrp)

plt.figure(figsize=(15,3))
plt.subplot(1,2,1)
plt.plot(mmas_lrp, color="green", label="mmas")
plt.plot(imax, mmas_lrp[imax], 'o', color="green")
plt.plot(mmenos_lrp, color="blue", label="mmenos")
plt.plot(imax, mmenos_lrp[imax], 'o', color="blue")
plt.axvline(imax, color="gray", alpha=0.2)
plt.legend()
plt.subplot(1,2,2)
plt.plot(epc_lrp, color="red", label="EPC")
plt.plot(imax, epc_lrp[imax], 'o', color="red")
plt.legend()
plt.show()

In [ ]:
imagen = 0
plt.figure(figsize=(15,3))
plt.subplot(1,2,1)
plt.plot(pred_mmas_gradientxinput[:, imagen, y_train[imagen]], '.-', label="$m^+$")
plt.plot(pred_mmenos_gradientxinput[:, imagen, y_train[imagen]], '.-', label="$m^-$")
plt.legend()
plt.subplot(1,2,2)
plt.plot(factor1 * (pred_mmas_gradientxinput[:, imagen, y_train[imagen]] - pred_mmenos_gradientxinput[:, imagen, y_train[imagen]]) / ACC, label="EPC")
plt.legend()
plt.show()

# X[Rx <= U] = 0 <--- Azul
# X[Rx >= U] = 0 <--- Naranja

In [ ]:
imagen = 0
plt.figure(figsize=(15,3))
plt.subplot(1,2,1)
plt.plot(pred_mmas_lrp[:, imagen, y_train[imagen]], '.-', label="$m^+$")
plt.plot(pred_mmenos_lrp[:, imagen, y_train[imagen]], '.-', label="$m^-$")
plt.legend()
plt.subplot(1,2,2)
plt.plot(factor1 * (pred_mmas_lrp[:, imagen, y_train[imagen]] - pred_mmenos_lrp[:, imagen, y_train[imagen]]) / ACC, label="EPC")
plt.legend()
plt.show()

# X[Rx <= U] = 0 <--- Azul
# X[Rx >= U] = 0 <--- Naranja

## <font color="#F48E16">Usando BLUR como baseline</font>

In [ ]:
model_name = "MOBILENET"
activation = "relu"
l1reg = 0.0
use_blur = True

R_gradientxinput = gradientxinput(model, x_train, y_train)
R_lrp = gradientxinput(model_lrp, x_train, y_train)

(mmas_gradientxinput_blur,
 mmenos_gradientxinput_blur,
 pred_mmas_gradientxinput_blur,
 pred_mmenos_gradientxinput_blur) = epc(x_train,
                                   y_train,
                                   model,
                                   R_gradientxinput.reshape(-1, 224*224*3),
                                   X_blur=x_train_blur,
                                   use_blur=use_blur,
                                   local=True,
                                   verbose=True)

# LRP
(mmas_lrp_blur,
 mmenos_lrp_blur,
 pred_mmas_lrp_blur,
 pred_mmenos_lrp_blur) = epc(x_train,
                        y_train,
                        model,
                        R_lrp.reshape(-1, 224*224*3),
                        X_blur=x_train_blur,
                        use_blur=use_blur,
                        local=True,
                        verbose=True)


print(" > DONE")

In [ ]:
at = 224*224*3
iterador_at = np.arange(0, at, 1506) # Voy de 1506 en 1506 para acelerar y tener 100 pasos
ar = at - iterador_at
factor1 = (at - ar) / at

In [ ]:
epc_gradientxinput_blur = factor1 * (mmas_gradientxinput_blur - mmenos_gradientxinput_blur) / ACC
imax = np.argmax(epc_gradientxinput_blur)

plt.figure(figsize=(15,3))
plt.subplot(1,2,1)
plt.plot(mmas_gradientxinput_blur, color="green", label="mmas")
plt.plot(imax, mmas_gradientxinput_blur[imax], 'o', color="green")
plt.plot(mmenos_gradientxinput_blur, color="blue", label="mmenos")
plt.plot(imax, mmenos_gradientxinput_blur[imax], 'o', color="blue")
plt.axvline(imax, color="gray", alpha=0.2)
plt.legend()
plt.subplot(1,2,2)
plt.plot(epc_gradientxinput_blur, color="red", label="EPC")
plt.plot(imax, epc_gradientxinput_blur[imax], 'o', color="red")
plt.legend()
plt.show()

In [ ]:
epc_lrp_blur = factor1 * (mmas_lrp_blur - mmenos_lrp_blur) / ACC
imax = np.argmax(epc_lrp_blur)

plt.figure(figsize=(15,3))
plt.subplot(1,2,1)
plt.plot(mmas_lrp_blur, color="green", label="mmas")
plt.plot(imax, mmas_lrp_blur[imax], 'o', color="green")
plt.plot(mmenos_lrp_blur, color="blue", label="mmenos")
plt.plot(imax, mmenos_lrp_blur[imax], 'o', color="blue")
plt.axvline(imax, color="gray", alpha=0.2)
plt.legend()
plt.subplot(1,2,2)
plt.plot(epc_lrp_blur, color="red", label="EPC")
plt.plot(imax, epc_lrp_blur[imax], 'o', color="red")
plt.legend()
plt.show()

In [ ]:
imagen = 0
plt.figure(figsize=(15,3))
plt.subplot(1,2,1)
plt.plot(pred_mmas_gradientxinput_blur[:, imagen, y_train[imagen]], '.-', label="$m^+$")
plt.plot(pred_mmenos_gradientxinput_blur[:, imagen, y_train[imagen]], '.-', label="$m^-$")
plt.legend()
plt.subplot(1,2,2)
plt.plot(factor1 * (pred_mmas_gradientxinput_blur[:, imagen, y_train[imagen]] - pred_mmenos_gradientxinput_blur[:, imagen, y_train[imagen]]) / ACC, label="EPC")
plt.legend()
plt.show()

# X[Rx <= U] = 0 <--- Azul
# X[Rx >= U] = 0 <--- Naranja

In [ ]:
imagen = 0
plt.figure(figsize=(15,3))
plt.subplot(1,2,1)
plt.plot(pred_mmas_lrp_blur[:, imagen, y_train[imagen]], '.-', label="$m^+$")
plt.plot(pred_mmenos_lrp_blur[:, imagen, y_train[imagen]], '.-', label="$m^-$")
plt.legend()
plt.subplot(1,2,2)
plt.plot(factor1 * (pred_mmas_lrp_blur[:, imagen, y_train[imagen]] - pred_mmenos_lrp_blur[:, imagen, y_train[imagen]]) / ACC, label="EPC")
plt.legend()
plt.show()

# X[Rx <= U] = 0 <--- Azul
# X[Rx >= U] = 0 <--- Naranja

In [ ]:
# Observaciones:
# blur vs black/gray/0 baseline cambia resultados
# hay dependencia fuerte del tipo de perturbación

# Antes, en MNIST, el valor 0 significaba "ausencia de información" y el -1/1 significaba "hay o debería haber algo aquí"
# Ahora, en Imagenet, colores oscuros tienen valores cercanos a -1 y colores claros colores cercanos a +1. Es decir, no
# significa más o menos información, sino que estamos codificando colores.
# Imagina la imagen de un gato gris (colores cercanos a 0). ¿El gato no tiene información? Claro que sí, esa interpretación no es buena.
# Por eso hacer 0 no es bueno en estos datasets tan grandes.